In [1]:
import pandas as pd, numpy as np, shutil, re, yaml
from pathlib import Path
from PIL import Image
inp, W = Path("/kaggle/input"), Path("/kaggle/working")

def block_split(keys, size, seed):
    blocks = [keys[i:i + size] for i in range(0, len(keys), size)]
    order = np.random.default_rng(seed).permutation(len(blocks)); n = len(blocks); out = {}
    for rank, b in enumerate(order):
        sp = "test" if rank < round(0.2 * n) else "val" if rank < round(0.3 * n) else "train"
        for k in blocks[b]: out[k] = sp
    return out

tz_src = next(p for p in inp.rglob("tz_tiles") if p.is_dir())
tz_lab = pd.read_csv(next(inp.rglob("tz_labels.csv")), dtype={"image": str})
num = lambda s: int(re.sub(r"\D", "", s) or 0)
photo_split = block_split(sorted(tz_lab.photo.unique(), key=num), 10, 42)
tile_split = tz_lab.drop_duplicates("image").set_index("image").photo.map(photo_split)

for sp in ["train", "val", "test"]:
    for sub in ["images", "labels"]: (W / "tz_split" / sp / sub).mkdir(parents=True, exist_ok=True)
for img, sp in tile_split.items():
    shutil.copy(tz_src / "test" / "images" / f"{img}.jpg", W / "tz_split" / sp / "images" / f"tz_{img}.jpg")
    shutil.copy(tz_src / "test" / "labels" / f"{img}.txt", W / "tz_split" / sp / "labels" / f"tz_{img}.txt")

pd.Series(photo_split, name="split").to_csv(W / "tz_photo_split.csv")
print("Photos per split:\n", pd.Series(photo_split).value_counts())
print("Tiles per split:\n", tile_split.value_counts())

Photos per split:
 train    430
test     111
val       70
Name: count, dtype: int64
Tiles per split:
 photo
train    3754
test     1040
val       659
Name: count, dtype: int64


In [2]:
import random
gh_src = next(p for p in inp.rglob("gh_images") if p.is_dir())
ids = sorted((p.stem for p in (gh_src / "test" / "images").glob("*.jpg")), key=lambda s: int(s.split("_")[1]))
gh_split = block_split(ids, 30, 43)
T, STEP = 640, 544
for sp in ["train", "val", "test"]:
    for sub in ["images", "labels"]: (W / "gh_split" / sp / sub).mkdir(parents=True, exist_ok=True)

for stem in ids:
    sp = gh_split[stem]
    im = Image.open(gh_src / "test" / "images" / f"{stem}.jpg").convert("RGB")
    if min(im.size) < T:                                          
        c = Image.new("RGB", (max(im.width, T), max(im.height, T)), (0, 0, 0)); c.paste(im, (0, 0)); im = c
    arr = np.asarray(im); H, Wd = arr.shape[:2]
    L = np.array([list(map(float, l.split())) for l in (gh_src / "test" / "labels" / f"{stem}.txt").read_text().splitlines()]).reshape(-1, 5)
    ow, oh = Image.open(gh_src / "test" / "images" / f"{stem}.jpg").size
    cls, cx, cy, bw, bh = L[:, 0].astype(int), L[:, 1] * ow, L[:, 2] * oh, L[:, 3] * ow, L[:, 4] * oh
    for x0 in sorted(set(list(range(0, Wd - T, STEP)) + [Wd - T])):
        for y0 in sorted(set(list(range(0, H - T, STEP)) + [H - T])):
            tile = arr[y0:y0 + T, x0:x0 + T]
            if (tile.mean(axis=2) < 40).mean() > 0.6: continue
            lines = []
            for k in np.where((cx >= x0) & (cx < x0 + T) & (cy >= y0) & (cy < y0 + T))[0]:
                x1, x2 = max(cx[k] - bw[k] / 2, x0), min(cx[k] + bw[k] / 2, x0 + T)
                y1, y2 = max(cy[k] - bh[k] / 2, y0), min(cy[k] + bh[k] / 2, y0 + T)
                lines.append(f"{cls[k]} {((x1+x2)/2-x0)/T:.6f} {((y1+y2)/2-y0)/T:.6f} {(x2-x1)/T:.6f} {(y2-y1)/T:.6f}")
            name = f"{stem}__{x0}_{y0}"
            Image.fromarray(tile).save(W / "gh_split" / sp / "images" / f"{name}.jpg", quality=92)
            (W / "gh_split" / sp / "labels" / f"{name}.txt").write_text("\n".join(lines))

train_imgs = sorted((W / "gh_split" / "train" / "images").glob("*.jpg"))
random.seed(0)
for p in random.sample(train_imgs, max(0, len(train_imgs) - 8000)):      # keep sites balanced in training
    p.unlink(); (W / "gh_split" / "train" / "labels" / f"{p.stem}.txt").unlink()

pd.Series(gh_split, name="split").to_csv(W / "gh_image_split.csv")
print("Images per split:\n", pd.Series(gh_split).value_counts())
for sp in ["train", "val", "test"]:
    print(sp, "tiles:", len(list((W / "gh_split" / sp / "images").glob("*.jpg"))))
!du -sh /kaggle/working/tz_split /kaggle/working/gh_split

Images per split:
 train    2113
test      600
val       330
Name: count, dtype: int64
train tiles: 8000
val tiles: 2150
test tiles: 3943
190M	/kaggle/working/tz_split
637M	/kaggle/working/gh_split
